# 001 · EarlySign Core Concepts: (1) Ledger
This tutorial notebook explains the concept of **ledger** that is at the core of `earlysign`.

For demonstrative purposes, we take a step-by-step approach, and we use the bare functionality provided by `earlysign.core`.
In `earlysign`, we also have higher-level abstractions such as in `earlysign.framework`, which will be used later in this tutorial series.

- In `earlysign`, **Ledger is the central concept**: every statistical step is a **write/read** to the ledger.
- Each event row has **payload** (data you write) and **attributes** (context), and the ledger assigns **timestamp/uuid** automatically.

In this notebook, we demonstrate the procedure of a group sequential test in an A/B-testing scenario.
- We plan **five looks**, and the total type-I error rate is controlled by appropriately deriving the boundaries.
- **Information time**: the **sample-size fraction** $t = n/N_{\max}$.
- **Boundary**: Brownian motion approximation, O’Brien–Fleming one-sided (variant B being better than A)
  $$ c(t) = \frac{z_{1-\alpha}}{\sqrt{t}}, \quad \alpha=0.05. $$
  The alpha level is $\alpha=0.05$. For the O'Brien–Fleming style boundary we use the **one-sided 95th percentile** $z_{1-\alpha}\approx 1.64485$, yielding:
$$ c(t) = \frac{1.64485}{\sqrt{t}}. $$
- **Decision rule**: one-sided improvement of B over A:
  $$ Z \ge c(t) \ \Rightarrow\ \text{“reject A ≥ B” (i.e., B is better)}. $$
  Otherwise, **continue** to the next observation step.

Per look we do: **(1) Observe → (2) Compute Z → (3) Info time → (4) Boundary → (5) Decide**, writing each result to the ledger and immediately displaying the full ledger row(s).

**Look 1** is shown **step-by-step** across separate cells. **Looks 2–5** run in **one cell** with the same steps performed inside a loop.

In [ ]:
import math

import ibis
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from earlysign.core.ledger import Ledger

# DataFrame visibility
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 0)
pd.set_option("display.max_columns", None)


# Utility
def show_new_and_context(ledger, prev_ts=None, context_tail=6):
    df = ledger.t.order_by("timestamp").execute()
    if prev_ts is None:
        return df.tail(context_tail)
    is_new = df["timestamp"] > prev_ts
    new_rows = df[is_new]
    if new_rows.empty:
        return df.tail(context_tail)
    context = df[~is_new].tail(context_tail)
    return pd.concat([context, new_rows], ignore_index=True)


from typing import List, Optional

from scipy.optimize import brentq
from scipy.stats import multivariate_normal, norm


class OBFSpendingSolver:
    def solve_for_critical_value(
        self,
        alpha: float,
        info_times: List[float],
        past_upper: List[float],
        spending_name: str = "obrien-fleming",
    ) -> Optional[float]:
        if spending_name != "obrien-fleming":
            raise NotImplementedError(f"Spending '{spending_name}' not implemented")
        if not info_times:
            return None
        j = len(past_upper) + 1
        assert 1 <= j <= len(info_times), "Invalid look index."
        C = self._corr_from_info_times(info_times[:j])
        target = 1.0 - self._obf_spending_one_sided(alpha, info_times[j - 1])

        def root(cj: float) -> float:
            cuts = np.array(past_upper + [cj], dtype=float)
            return self._mvn_rect_prob(cuts, C[:j, :j]) - target

        lo, hi = -1.0, 8.0
        f_lo, f_hi = root(lo), root(hi)
        k = 0
        while f_lo * f_hi > 0 and k < 30:
            lo -= 1.0
            hi += 1.0
            f_lo, f_hi = root(lo), root(hi)
            k += 1
        return float(brentq(root, lo, hi, xtol=1e-6, maxiter=400))

    @staticmethod
    def _corr_from_info_times(t: List[float]) -> np.ndarray:
        t = np.asarray(t, dtype=float)
        n = len(t)
        C = np.ones((n, n), dtype=float)
        for i in range(n):
            for j in range(n):
                C[i, j] = np.sqrt(min(t[i], t[j]) / max(t[i], t[j]))
        return C

    @staticmethod
    def _obf_spending_one_sided(alpha: float, t: float) -> float:
        z = float(norm.isf(alpha))
        t = max(float(t), 1e-12)
        return 1.0 - float(norm.cdf(z / np.sqrt(t)))

    @staticmethod
    def _mvn_rect_prob(cuts: np.ndarray, cov: np.ndarray) -> float:
        try:
            return float(
                multivariate_normal.cdf(
                    x=np.array(cuts, float),
                    mean=np.zeros(len(cuts)),
                    cov=cov,
                    maxpts=200000,
                )
            )
        except Exception:
            rng = np.random.default_rng(20250922)
            L = np.linalg.cholesky(cov + 1e-12 * np.eye(cov.shape[0]))
            M = 300000
            Z = rng.standard_normal((M, cov.shape[0])) @ L.T
            return float((Z < np.array(cuts)).all(axis=1).mean())


# Type classes for ledger tagging
class Design(dict):
    pass


class Observation(dict):
    pass


class WaldZ(dict):
    pass


class InfoTime(dict):
    pass


class GSTBoundary(dict):
    pass


class Decision(dict):
    pass


class DecisionSignal(dict):
    pass


class Log(dict):
    pass

### Demo Setup

In [ ]:
class DataGenerator:
    # True rates (B slightly better)
    pA = 0.15
    pB = 0.17

    def __init__(self, seed=20250921):
        # Reproducibility
        self.rng = np.random.default_rng(seed)

    def sample(self, nA: int, nB: int):
        response_A = self.rng.binomial(1, self.pA, size=nA)
        response_B = self.rng.binomial(1, self.pB, size=nB)
        return int(response_A.sum()), int(response_B.sum())


data_generator = DataGenerator()

## Create the Ledger

In this `earlysign` library, we make an extensive use of the concept of **ledger**.

The ledger table has the following columns (`timestamp` and `uuid` are auto-managed by the `Ledger` class):
- `timestamp`: timestamp
- `uuid`: unique identifier
- `type`: a short tag for the event type
- `payload`: an arbitrary json storing the event data
- `attributes`: a dictionary storing contextual labels

We create an in-memory DuckDB connection and ensure the ledger table exists.

In [ ]:
con = ibis.duckdb.connect(":memory:")
ledger = Ledger(con, "events")
ledger.ensure()

# Empty schema view (all columns)
ledger.t.limit(0).execute()

## Register the experiment design

We store the design as one event in the ledger: the payload describes the test,
and labels mark the experiment id. We keep the design simple and **fixed** for this tutorial.

In [ ]:
alpha = 0.05
z_one_sided = 1.6448536269514722
looks = [0.40, 0.50, 0.70, 1.00]
Nmax_per_arm = 1000
n_plan = [int(t * Nmax_per_arm) for t in looks]

design = {
    "test_name": "AB_binomial_rate_diff",
    "alpha": alpha,
    "sides": "one-sided (B better than A)",
    "looks": looks,
    "planned_n_per_look": n_plan,
    "Nmax_per_arm": Nmax_per_arm,
    "metric": "conversion",
    "variant_labels": {"A": "control", "B": "treatment"},
}
attributes = {"experiment": "demo_core_001"}

ledger.insert(data=Design(design), attributes=attributes)
ledger.t.filter(ledger.t.type == "Design").order_by("timestamp").execute()

You can see that your crafted payload has been registered into the ledger, along with the auto-recorded record ID and timestamp.

# Look 1 — step-by-step

We now execute the first observation and the first look.
As far as this tutorial is concerned, the information time refers to the ratio of the current sample size to the maximum planned sample size.


### Step 1. Observe and write observation

We add observations up to the planned first look size and write the cumulative counts as-is in the **payload**.
This tutorial keeps payloads simple and directly readable.

In [ ]:
cum = {"A": {"n": 0, "m": 0}, "B": {"n": 0, "m": 0}}

target = n_plan[0]
prev_ts = ledger.t.timestamp.max().execute()

addA = target - cum["A"]["n"]
addB = target - cum["B"]["n"]

mA, mB = data_generator.sample(addA, addB)

cum["A"]["n"] += addA
cum["B"]["n"] += addB
cum["A"]["m"] += mA
cum["B"]["m"] += mB

ledger.insert(
    data=Observation(
        {
            "look": 1,
            "A": {"success": cum["A"]["m"], "n": cum["A"]["n"]},
            "B": {"success": cum["B"]["m"], "n": cum["B"]["n"]},
        }
    ),
    attributes={**attributes, "look": 1},
)

show_new_and_context(ledger, prev_ts)

### Step 2. Compute Wald Z and write

We compute the Wald Z for the difference in proportions \(B-A\):
$$
Z = \frac{\hat p_B - \hat p_A}{\sqrt{\hat p_A(1-\hat p_A)/n_A + \hat p_B(1-\hat p_B)/n_B}}.
$$
We then store it with the look label.



In [ ]:
def wald_z_from_counts(mA, nA, mB, nB, eps=1e-12):
    pAhat = mA / max(nA, eps)
    pBhat = mB / max(nB, eps)
    var = pAhat * (1 - pAhat) / max(nA, eps) + pBhat * (1 - pBhat) / max(nB, eps)
    if var <= 0:
        return float("nan")
    return (pBhat - pAhat) / math.sqrt(var)


prev_ts = ledger.t.timestamp.max().execute()
z1 = wald_z_from_counts(cum["A"]["m"], cum["A"]["n"], cum["B"]["m"], cum["B"]["n"])
ledger.insert(
    data=WaldZ({"look": 1, "wald_z": z1}),
    attributes={**attributes, "look": 1},
)

show_new_and_context(ledger, prev_ts)

You can see that we have added the first computed statistic value to the ledger now.

### Step 3. Compute information time, O’Brien–Fleming boundary, and write them

Information time \(t\) is the **sample-size ratio** (average over arms):
$$ t = \frac{1}{2}\left(\frac{n_A + n_B}{N_{\max}}\right). $$
We compute it and store it in the ledger.

Then, reading the current information time from the ledger, we compute the boundary for this look.

Using the Brownian approximation with an O’Brien–Fleming one-sided design,
$$ c(t) = \frac{z_{1-\alpha}}{\sqrt{t}}. $$
We store the boundary as a pair `(upper=c, lower=-c)` on the **Z-scale**.

In [ ]:
# === Step 3: Threshold computation (InfoTime + Boundary in one cell; upper-only) ===
# InfoTime uses the sample-size ratio: t = ((n_A + n_B)/2) / Nmax_per_arm
# Then solve the OBF upper boundary for the *new* look and store it. All via the Ledger.

import numpy as np
from IPython.display import display


# --- helpers for this step (defined where first needed) ---
def _read_design(ledger):
    recs = (
        ledger.t.filter(ledger.t.type == "Design")
        .order_by("timestamp")
        .execute()
        .to_dict("records")
    )
    if not recs:
        raise RuntimeError("Design record not found.")
    return recs[-1]["payload"]


def _read_info_times(ledger):
    rows = (
        ledger.t.filter(ledger.t.type == "InfoTime")
        .order_by("timestamp")
        .execute()
        .to_dict("records")
    )
    pairs = [(r["payload"].get("look"), r["payload"].get("t")) for r in rows]
    seen = set()
    looks_t = []
    for lk, t in pairs:
        if lk is None or t is None or lk in seen:
            continue
        lk = int(lk)
        seen.add(lk)
        looks_t.append((lk, float(t)))
    looks_t.sort(key=lambda x: x[0])
    return [t for _, t in looks_t]


def _read_past_upper(ledger):
    rows = (
        ledger.t.filter(ledger.t.type == "GSTBoundary")
        .order_by("timestamp")
        .execute()
        .to_dict("records")
    )
    out = {}
    for r in rows:
        p = r.get("payload", {}) or {}
        lk = p.get("look")
        up = p.get("upper")
        if lk is not None and up is not None:
            out[int(lk)] = float(up)
    return out


# 1) Read design
design = _read_design(ledger)
alpha = float(design.get("alpha"))
spending_name = (design.get("spending") or {}).get("name", "obrien-fleming")
Nmax_per_arm = float(design.get("Nmax_per_arm"))
attributes = locals().get("attributes", {})

# 2) Compute InfoTime for next look (sample-size ratio). Replace n_A, n_B with your live counters if available.
info_times_now = _read_info_times(ledger)
j_next = len(info_times_now) + 1

# If your environment exposes n_A, n_B, use them. Otherwise keep this example fallback:
if "n_A" in globals() and "n_B" in globals():
    nA, nB = float(n_A), float(n_B)
else:
    # Demo progression: grows with look index; replace in production
    nA, nB = j_next * 200.0, j_next * 200.0

t_next = ((nA + nB) / 2.0) / Nmax_per_arm
t_next = max(1e-6, min(1.0, t_next))

# 3) Insert InfoTime(look=j_next, t_next)
prev_ts = ledger.t.timestamp.max().execute()
ledger.insert(
    data=InfoTime({"look": j_next, "t": float(t_next)}),
    attributes={**attributes, "look": j_next},
)
if "show_new_and_context" in globals():
    display(show_new_and_context(ledger, prev_ts))
prev_ts = ledger.t.timestamp.max().execute()

# 4) Re-read InfoTime & past boundaries; solve only c_j_next using the solver
info_times = _read_info_times(ledger)  # includes t_next
past_map = _read_past_upper(ledger)  # {look: upper}
past_upper = [past_map[i] for i in range(1, j_next) if i in past_map]

solver = OBFSpendingSolver()
c_j = solver.solve_for_critical_value(
    alpha=alpha,
    info_times=info_times[:j_next],
    past_upper=past_upper,
    spending_name=spending_name,
)

# 5) Insert boundary for j_next (upper only; no lower)
ledger.insert(
    data=GSTBoundary({"look": j_next, "upper": float(c_j), "scale": "z"}),
    attributes={**attributes, "look": j_next},
)
if "show_new_and_context" in globals():
    display(show_new_and_context(ledger, prev_ts))

print(f"[Threshold] look {j_next}: t={info_times[j_next - 1]:.6f}, c={c_j:.6f}")

We now have the threshold to compare the statistic value against.

### Step 4. Make decision and write

One-sided improvement of B over A: **reject** when $Z \ge c(t)$.
Otherwise, **continue**.


In [ ]:
# === Step 4: Decision (one-sided, upper-only) ===
# Compare your Z for the latest look j with the corresponding boundary c_j from the Ledger.

import numpy as np
from IPython.display import display


def _read_latest_boundary(ledger):
    rows = (
        ledger.t.filter(ledger.t.type == "GSTBoundary")
        .order_by("timestamp")
        .execute()
        .to_dict("records")
    )
    if not rows:
        raise RuntimeError("No GSTBoundary found. Run the threshold step first.")
    pairs = []
    for r in rows:
        p = r.get("payload", {}) or {}
        lk = p.get("look")
        up = p.get("upper")
        if lk is not None and up is not None:
            pairs.append((int(lk), float(up)))
    pairs.sort(key=lambda x: x[0])
    return pairs[-1]  # (j, c)


# fetch latest c_j
j, c = _read_latest_boundary(ledger)

# Expect a statistic z (compute earlier in your pipeline). If you name them z1,z2,... map it here:
if "z" not in globals() or globals()["z"] is None:
    z = globals().get(f"z{j}")
else:
    z = globals()["z"]
if z is None:
    raise RuntimeError(f"Z statistic for look {j} is not available in this scope.")

prev_ts = ledger.t.timestamp.max().execute()
if np.isfinite(z) and (z >= c):
    signal, reason = "reject_A_ge_B", f"z={z:.3f} >= c={c:.3f}"
else:
    signal, reason = "continue", f"z={z:.3f} < c={c:.3f}"

ledger.insert(
    data=DecisionSignal({"look": j, "signal": signal, "reason": reason}),
    attributes={**locals().get("attributes", {}), "look": j},
)
if "show_new_and_context" in globals():
    display(show_new_and_context(ledger, prev_ts))
print(f"[Decision look {j}] z={z:.4f}, c={c:.4f} -> {signal}")

At this time, the signal says we should continue to the next look, instead of rejecting the null.

So far, we have demonstrated one set of procedures (one look) of interim analysis:
- `Observation` → counts up to look 1.
- `WaldZ` → interim statistic.
- `InfoTime` → information time \(t\).
- `GSTBoundary` → O’Brien–Fleming upper boundary on Z-scale.
- `DecisionSignal` → interim decision.

All results were **written** in the ledger and they were used in the subsequent calculations/decisions.

Now, we will repeat these steps according to the predetermined design.

# Remaining Looks — same steps inside a single loop

We now execute the rest of the experiment up to the final look at **t ≈ 1.00**.

In [ ]:
prev_ts = ledger.t.timestamp.max().execute()

for lk in range(2, len(looks) + 1):
    # Step 1: Observe
    target = n_plan[lk - 1]
    addA = target - cum["A"]["n"]
    addB = target - cum["B"]["n"]
    mA, mB = data_generator.sample(addA, addB)
    cum["A"]["n"] += addA
    cum["B"]["n"] += addB
    cum["A"]["m"] += mA
    cum["B"]["m"] += mB
    obs = {
        "look": lk,
        "A": {"success": cum["A"]["m"], "n": cum["A"]["n"]},
        "B": {"success": cum["B"]["m"], "n": cum["B"]["n"]},
    }
    ledger.insert(data=Observation(obs), attributes={**attributes, "look": lk})

    # Step 2: Z
    z = ((cum["B"]["m"] / cum["B"]["n"]) - (cum["A"]["m"] / cum["A"]["n"])) / math.sqrt(
        (cum["A"]["m"] / cum["A"]["n"])
        * (1 - cum["A"]["m"] / cum["A"]["n"])
        / cum["A"]["n"]
        + (cum["B"]["m"] / cum["B"]["n"])
        * (1 - cum["B"]["m"] / cum["B"]["n"])
        / cum["B"]["n"]
    )
    ledger.insert(
        data=WaldZ({"look": lk, "wald_z": z}),
        attributes={**attributes, "look": lk},
    )

    # Step 3: info time and boundary
    t = 0.5 * (cum["A"]["n"] / Nmax_per_arm + cum["B"]["n"] / Nmax_per_arm)
    ledger.insert(
        data=InfoTime({"look": lk, "t": t}),
        attributes={**attributes, "look": lk},
    )

    info_times = _read_info_times(ledger)  # includes t_next
    past_map = _read_past_upper(ledger)  # {look: upper}
    past_upper = [past_map[i] for i in range(1, len(past_map) + 1) if i in past_map]

    solver = OBFSpendingSolver()
    c = solver.solve_for_critical_value(
        alpha=alpha,
        info_times=info_times,
        past_upper=past_upper,
        spending_name=spending_name,
    )
    ledger.insert(
        data=GSTBoundary({"look": lk, "upper": c, "scale": "z"}),
        attributes={**attributes, "look": lk},
    )

    # Step 4: decision
    if np.isfinite(z) and (z >= c):
        dec = {
            "look": lk,
            "signal": "reject_A_ge_B",
            "reason": f"z={z:.3f} >= c={c:.3f}",
        }
    else:
        dec = {"look": lk, "signal": "continue", "reason": f"z={z:.3f}, c={c:.3f}"}
    ledger.insert(data=DecisionSignal(dec), attributes={**attributes, "look": lk})

    if "reject" in dec["signal"]:
        ledger.insert(
            data=Log(
                {
                    "message": "We terminate the experiment early in light of the sequential testing procedure. The null hypothesis has been rejected. Let's go with the alternative variant!"
                }
            ),
        )

display(show_new_and_context(ledger, prev_ts))

# Consolidated reporting

### Summarize looks

In [ ]:
# Source of truth: the ledger
info = {
    r["payload"]["look"]: r["payload"]["t"]
    for r in ledger.t.filter(ledger.t.type == "InfoTime").execute().to_dict("records")
}
wald = {
    r["payload"]["look"]: r["payload"]["wald_z"]
    for r in ledger.t.filter(ledger.t.type == "WaldZ").execute().to_dict("records")
}
crit = {
    r["payload"]["look"]: r["payload"]["upper"]
    for r in ledger.t.filter(ledger.t.type == "GSTBoundary")
    .execute()
    .to_dict("records")
}
dec = {
    r["payload"]["look"]: r["payload"]
    for r in ledger.t.filter(ledger.t.type == "DecisionSignal")
    .execute()
    .to_dict("records")
}

rows = []
for lk in sorted(info.keys()):
    rows.append(
        {
            "look": lk,
            "info_time": info[lk],
            "wald_z": wald[lk],
            "crit_upper": crit[lk],
            "decision": dec[lk]["signal"],
            "reason": dec[lk]["reason"],
        }
    )
pd.DataFrame(rows).sort_values("look")

### Plot Z trajectory vs O’Brien–Fleming boundary

In [ ]:
tvals = [info[k] for k in sorted(info.keys())]
zvals = [wald[k] for k in sorted(wald.keys())]
cvals = [crit[k] for k in sorted(crit.keys())]

plt.figure(figsize=(6, 4))
plt.plot(
    tvals, zvals, marker="o", linewidth=2.0, label="Wald's Z Statistic", color="#d55e00"
)
plt.plot(
    tvals, cvals, marker="o", linewidth=1.0, linestyle=":", color="C0", label="Boundary"
)
plt.xlim(-0.1, 1.1)
plt.ylim(-0.1, None)
plt.xlabel("Information fraction")
plt.ylabel("Z")
plt.axhline(0, color="gray", linewidth=1, linestyle="-")
plt.title("Interim Z vs. O'Brien–Fleming one-sided boundary (B better)")
plt.legend()
plt.show()

### Full ledger (recent context retained)

In [ ]:
# Show the last ~25 rows for audit (all columns intact)
df_all = ledger.t.order_by("timestamp").execute()
df_all.tail(25)

# Closing notes and next steps

- We recorded all steps as immutable events in the ledger.
- After each write, we immediately inspected the full ledger rows for that look.
- This demonstrated a **two-look one-sided** sequential test for B's improvement using only `earlysign.core`.
- In higher layers (`framework`, `templates`, `api`), the same idea remains: **everything is ledger operations**.


## O'Brien–Fleming function shape and spending matching

- **Analytic curve (idealized)**: \(c(t)=z/\sqrt{t}\) with \(z=\Phi^{-1}(1-\alpha)\).  
- **Spending-matched discrete points**: the notebook **solves \(c_j\)** so that
  \(\Pr(Z_1<c_1,\dots,Z_j<c_j) = 1 - A(t_j)\) where \(A(t)=1-\Phi(z/\sqrt{t})\).
  This enforces one-sided **Lan–DeMets OBF** spending exactly at recorded information times.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm

# Read alpha
drec = (
    ledger.t.filter(ledger.t.type == "Design")
    .order_by("timestamp")
    .execute()
    .to_dict("records")
)
alpha = float(drec[-1]["payload"]["alpha"])

# Info times
rows_t = (
    ledger.t.filter(ledger.t.type == "InfoTime")
    .order_by("timestamp")
    .execute()
    .to_dict("records")
)
pairs_t = [(r["payload"].get("look"), r["payload"].get("t")) for r in rows_t]
looks_t = [(int(lk), float(t)) for lk, t in pairs_t if lk is not None and t is not None]
looks_t.sort(key=lambda x: x[0])
ts_list = [t for _, t in looks_t]

# Boundaries
rows_b = (
    ledger.t.filter(ledger.t.type == "GSTBoundary")
    .order_by("timestamp")
    .execute()
    .to_dict("records")
)
pairs_b = [
    (int(r["payload"]["look"]), float(r["payload"]["upper"]))
    for r in rows_b
    if r.get("payload", {}).get("look") is not None
    and r["payload"].get("upper") is not None
]
pairs_b.sort(key=lambda x: x[0])
js = [j for j, _ in pairs_b]
cs = [c for _, c in pairs_b]

# Analytic curve
z_crit = float(norm.isf(alpha))
tgrid = np.linspace(0.02, 1.0, 200)
cgrid = z_crit / np.sqrt(tgrid)

plt.figure(figsize=(6, 4))
plt.plot(tgrid, cgrid, label="Analytic c(t)=z/√t")
if js:
    plt.scatter([ts_list[j - 1] for j in js], cs, label="Discrete (Lan–DeMets matched)")
plt.xlabel("Information fraction t")
plt.ylabel("Critical Z (upper)")
plt.title("O'Brien–Fleming: Analytic vs. Spending-matched Points")
plt.legend()
plt.show()